# 03 — Offsets, tenors and settlement dates

Three tools that stack: the `BDay` object so you can write `date + BDay(3)`, the tenor
grammar for `"1Y+2B"`, and settlement lags for "when does this actually pay?".

In [1]:
from datetime import date, datetime

import numpy as np
import pandas as pd

import better_calendar as bcal
from better_calendar import BDay, Calendar, Roll

## 1. `BDay`: the offset as an object

In [2]:
print(date(2026, 7, 31) + BDay(1))                   # Friday -> Monday
print(date(2026, 8, 3) - BDay(1))
print("2026-07-02" + BDay(1, cal="XNYS"))            # skips 3 July
print(date(2026, 7, 31) + BDay(1) * 5)               # multipliable
print(-BDay(2))

2026-08-03
2026-07-31
2026-07-06
2026-08-07
BDay(n=-2, cal=None, roll=<Roll.FOLLOWING: 'following'>)


It works on **every** input type, and on pandas containers:

In [3]:
series = pd.Series(pd.DatetimeIndex(["2026-07-31", "2026-08-03", "2026-12-24"]))
result = series + BDay(3, cal="XNYS")
pd.DataFrame({"traded": series, "+3 NYSE business days": result})

,traded,+3 NYSE business days
0,2026-07-31,2026-08-05
1,2026-08-03,2026-08-06
2,2026-12-24,2026-12-30


Containers are the delicate part. When you write `series + BDay(3)`, pandas does **not**
hand you the Series: it unwraps the underlying array, calls `__radd__` with that, then
rebuilds the container from whatever comes back. So the return value has to be something
pandas can wrap, **and** it has to carry the timezone if the input had one.

For a bare numpy array you additionally have to opt out of the ufunc machinery, or the
operation dies inside numpy.

In [4]:
array = np.array(["2026-07-31", "2026-08-03"], dtype="datetime64[D]")
print("numpy   :", (array + BDay(1)).strftime("%Y-%m-%d").tolist())

index = pd.DatetimeIndex(["2026-07-31"]).tz_localize("Europe/Paris")
shifted = index + BDay(1, cal=Calendar("paris", tz="Europe/Paris"))
print("tz-aware:", shifted.strftime("%Y-%m-%d %H:%M %Z").tolist(), "|", shifted.dtype)

numpy   : ['2026-08-03', '2026-08-04']
tz-aware: ['2026-08-03 00:00 CEST'] | datetime64[ns, Europe/Paris]


In practice `cal.offset(series, 3)` stays the recommended form: same answer, shorter path,
no operator dispatch to reason about. `BDay` is for the places where an offset **object**
reads better — a default argument, a configuration value.

In [5]:
nyse = bcal.get("XNYS")
pd.testing.assert_series_equal(series + BDay(3, cal="XNYS"), pd.Series(nyse.offset(series, 3)))
print("both forms give the same result")

both forms give the same result


### pandas interop: `to_pandas_offset` and where it diverges

For pandas machinery that insists on a genuine `DateOffset` (`date_range`, `resample`):

In [6]:
pandas_offset = nyse.to_pandas_offset()
pd.date_range("2026-07-01", "2026-07-10", freq=pandas_offset)

DatetimeIndex(['2026-07-01', '2026-07-02', '2026-07-06', '2026-07-07',
               '2026-07-08', '2026-07-09', '2026-07-10'],
              dtype='datetime64[ns]', freq='C')

**A real divergence, worth knowing**: `to_pandas_offset()` and `cal.offset()` do not agree
when the starting date is **not** a business day. We normalise then move (the
`numpy.busday_offset` semantics); pandas counts the normalisation as the move.

In [7]:
rows = []
for start in ("2026-07-31", "2026-07-02", "2026-08-01", "2026-07-03"):
    rows.append(
        {
            "start": start,
            "business day": nyse.is_bday(start),
            "cal.offset(d, 1)": nyse.offset(start, 1),
            "d + CustomBusinessDay": str((pd.Timestamp(start) + pandas_offset).date()),
        }
    )
table = pd.DataFrame(rows)
table["agree"] = table["cal.offset(d, 1)"] == table["d + CustomBusinessDay"]
table

,start,business day,"cal.offset(d, 1)",d + CustomBusinessDay,agree
0,2026-07-31,True,2026-08-03,2026-08-03,True
1,2026-07-02,True,2026-07-06,2026-07-06,True
2,2026-08-01,False,2026-08-04,2026-08-03,False
3,2026-07-03,False,2026-07-07,2026-07-06,False


Neither is wrong. But mixing the two in one pipeline ends badly, so start from a business
day, or stay on `cal.offset` and be explicit about `roll`.

### The `.cal` accessor

Importing the integration module registers an accessor on `Series` and `Index`:

In [8]:
import better_calendar.integrations.pandas_  # registers .cal

trades = pd.DataFrame({"traded": pd.to_datetime(["2026-07-02", "2026-08-01", "2026-12-24"])})
trades["business day"] = trades["traded"].cal.is_bday("XNYS")
trades["settles"] = trades["traded"].cal.offset(2, "XNYS")
trades["month later"] = trades["traded"].cal.add_tenor("1M", "XNYS", roll="MF")
trades

,traded,business day,settles,month later
0,2026-07-02,True,2026-07-07,2026-08-03
1,2026-08-01,False,2026-08-05,2026-09-01
2,2026-12-24,True,2026-12-29,2027-01-25


## 2. Tenors

Grammar: `term (('+' | '-') term)*` where a term is `[-] INT unit`, with the units `D`
(calendar days), `B` (business days), `W` (weeks), `M` (months), `Y` (years).

In [9]:
start = "2026-07-31"     # a Friday
pd.DataFrame(
    [{"tenor": t, "result": bcal.add_tenor(start, t, cal="XNYS")}
     for t in ("1D", "3D", "1W", "2W", "1B", "5B", "1M", "3M", "1Y", "1Y+2B", "1M-1B")]
).set_index("tenor")

,result
tenor,
1D,2026-08-01
3D,2026-08-03
1W,2026-08-07
2W,2026-08-14
1B,2026-08-03
5B,2026-08-07
1M,2026-08-31
3M,2026-10-31
1Y,2027-07-31


`parse_tenor` exposes the parsed structure, useful for validating a configuration before
running it — and the result is memoised, because tenors arrive from config files inside
hot loops.

In [10]:
parsed = bcal.parse_tenor("1Y-2B")
print("terms           :", parsed.terms)
print("needs a calendar:", parsed.needs_calendar)
print("memoised        :", bcal.parse_tenor("1Y-2B") is parsed)
print("no B term       :", bcal.parse_tenor("3M").needs_calendar)

terms           : (TenorTerm(count=1, unit='Y'), TenorTerm(count=-2, unit='B'))
needs a calendar: True
memoised        : True
no B term       : False


### The two month-end rules, which must never be conflated

This is where the off-by-one-day bugs live.

- **Clamping** is unconditional: 31 January plus one month is 28 February, because the 31st
  of February does not exist. Nothing optional about it.
- **The end-of-month rule** is opt-in (`eom=True`): if the starting date is the **last day**
  of its month, the result is the last day of the target month.

In [11]:
cases = ["2026-01-31", "2026-02-28", "2026-04-30", "2026-02-27", "2026-07-15", "2024-02-29"]
pd.DataFrame(
    [
        {
            "start": d,
            "month end?": pd.Timestamp(d).is_month_end,
            "+1M": bcal.add_tenor(d, "1M"),
            "+1M (eom=True)": bcal.add_tenor(d, "1M", eom=True),
        }
        for d in cases
    ]
).set_index("start")

,month end?,+1M,+1M (eom=True)
start,,,
2026-01-31,True,2026-02-28,2026-02-28
2026-02-28,True,2026-03-28,2026-03-31
2026-04-30,True,2026-05-30,2026-05-31
2026-02-27,False,2026-03-27,2026-03-27
2026-07-15,False,2026-08-15,2026-08-15
2024-02-29,True,2024-03-29,2024-03-31


The last two rows show that the EOM rule fires **only** when the start is a month end: the
27th of February and the 15th of July do not move.

Clamping, for its part, is not reversible — and that is intended:

In [12]:
forward = bcal.add_tenor("2026-01-31", "1M")
back = bcal.add_tenor(forward, "-1M")
print(f"2026-01-31 +1M -> {forward} ; then -1M -> {back}")
print("information was lost at the clamp, which is the correct behaviour")

2026-01-31 +1M -> 2026-02-28 ; then -1M -> 2026-01-28
information was lost at the clamp, which is the correct behaviour


### Terms apply left to right

`"1M+2B"` is **not** `"2B+1M"` in general: adding a month to a Friday and adding a month to
the following Tuesday do not land in the same week.

In [13]:
start = "2026-01-30"   # a Friday
print(f'{start}  "1M+2B" -> {bcal.add_tenor(start, "1M+2B")}')
print(f'{start}  "2B+1M" -> {bcal.add_tenor(start, "2B+1M")}')

2026-01-30  "1M+2B" -> 2026-03-04
2026-01-30  "2B+1M" -> 2026-03-03


A syntax error points at the offending fragment:

In [14]:
for bad in ("3Q", "3M5D", "1.5M"):
    try:
        bcal.add_tenor("2026-01-01", bad)
    except bcal.TenorParseError as exc:
        print(f"{bad!r:10s} -> {exc}")

'3Q'       -> Cannot parse tenor '3Q': unknown unit at '3Q'. Expected terms like '3M', '2B', '-1Y+2B' with units D, B, W, M or Y.
'3M5D'     -> Cannot parse tenor '3M5D': expected '+' or '-' between terms at '5D'. Expected terms like '3M', '2B', '-1Y+2B' with units D, B, W, M or Y.
'1.5M'     -> Cannot parse tenor '1.5M': unknown unit at '1.5M'. Expected terms like '3M', '2B', '-1Y+2B' with units D, B, W, M or Y.


A tenor's default roll is `NONE`: a tenor is a **period**, and adjusting it is a separate
decision you take explicitly.

In [15]:
print("raw        :", bcal.add_tenor("2026-04-30", "1M", eom=True))                    # a Sunday
print("adjusted MF:", bcal.add_tenor("2026-04-30", "1M", eom=True, roll=Roll.MODIFIED_FOLLOWING))

raw        : 2026-05-31
adjusted MF: 2026-05-29


## 3. Settlement dates

`spot(d, currency)` answers "if I trade today, when does it settle". The default calendar
comes from the currency through the alias table.

In [16]:
pd.DataFrame(
    [
        {
            "currency": c,
            "lag (business days)": bcal.spot_lag(c),
            "calendar": bcal.get(c).name,
            "spot from 2026-07-31": bcal.spot("2026-07-31", c),
        }
        for c in ("EUR", "USD", "GBP", "CAD", "JPY", "CHF", "TRY")
    ]
).set_index("currency")

,lag (business days),calendar,spot from 2026-07-31
currency,,,
EUR,2,fin:TARGET2,2026-08-04
USD,2,fin:NYB,2026-08-04
GBP,0,fin:LNB,2026-07-31
CAD,1,fin:TRB,2026-08-04
JPY,2,fin:TKB,2026-08-04
CHF,2,fin:ZUB,2026-08-04
TRY,0,ql:Turkey,2026-07-31


Sterling settles same day (T+0): these are **money-market / deposit** conventions, not FX
spot conventions — which are a property of the *pair*, not of a single currency.

The Canadian dollar is T+1 but lands on 4 August: 3 August is the Civic Holiday in Toronto.
The calendar does its job.

In [17]:
print("CAD T+1 from Friday 31 July :", bcal.spot("2026-07-31", "CAD"))
print("is 3 August a Toronto business day?", bcal.get("CAD").is_bday("2026-08-03"))

CAD T+1 from Friday 31 July : 2026-08-04
is 3 August a Toronto business day? False


For a cross-currency trade that has to settle in **two** centres at once, pass the
composite:

In [18]:
both_centres = bcal.get("EUR") & bcal.get("USD")
print("calendar   :", both_centres.name)
print("EUR alone  :", bcal.spot("2026-07-01", "EUR"))
print("EUR & USD  :", bcal.spot("2026-07-01", "EUR", cal=both_centres), " (skips 3 July)")

calendar   : (fin:TARGET2 & fin:NYB)
EUR alone  : 2026-07-03
EUR & USD  : 2026-07-06  (skips 3 July)


The lag table is a data file, not code — a desk whose convention differs corrects one row
without a release. It is read-only at runtime, because mutating it would silently move
every later settlement date.

In [19]:
print("known currencies :", ", ".join(sorted(bcal.SPOT_LAG)))
try:
    bcal.SPOT_LAG["EUR"] = 99
except TypeError as exc:
    print("\nmutation refused :", exc)

known currencies : AUD, CAD, CHF, CZK, DKK, EUR, GBP, HKD, HUF, JPY, MXN, NOK, NZD, PLN, SEK, SGD, TRY, USD, ZAR

mutation refused : 'mappingproxy' object does not support item assignment


## Recap

| Call | Role |
|---|---|
| `BDay(n, cal=, roll=)` | offset object, `d + BDay(3)` |
| `cal.offset(series, n)` | the recommended form for a container |
| `cal.to_pandas_offset()` | a real `DateOffset` for `date_range` / `resample` |
| `.cal` accessor | `series.cal.offset(2, "XNYS")` |
| `add_tenor(d, "1Y+2B", eom=)` | tenor grammar, left to right |
| `spot(d, ccy, cal=)` | settlement date |
| `SPOT_LAG`, `spot_lag(ccy)` | the lag table |

**Next:** [04 — Recurrences and schedules](04-recurrences-and-schedules.ipynb)